# 02 — Test Configuration & First GRPO Training (Unsloth)

Versione accelerata del notebook 01 con **Unsloth** (2-5x training più veloce, ~50-70% meno VRAM).

> **⚠️ Requisiti:** `bash setup.sh` su Colab (pinna anche `trl==0.22.2` e `transformers==4.56.2`)

1. **Unsloth import first** — importa `FastLanguageModel` prima di tutto
2. **Imports & GPU** — verifica che torch veda la GPU
3. **Dataset** — genera (o carica) il dataset sintetico (solo JSON)
4. **Model** — carica il modello con Unsloth + LoRA + 4-bit
5. **Rewards** — test rapido delle reward functions (JSON)
6. **Inference test** — genera un paio di completions e valuta
7. **GRPO Training** — avvia un mini-train (pochi step) con Unsloth

## 1. Setup su Colab

In [ ]:
!rm -rf GRPO-strict-generation
!git clone https://github.com/GiuseppeBellamacina/GRPO-strict-generation.git
%cd GRPO-strict-generation
!bash setup.sh

Cloning into 'tris'...
remote: Enumerating objects: 205, done.
remote: Counting objects: 100% (205/205), done.
remote: Compressing objects: 100% (130/130), done.
remote: Total 205 (delta 99), reused 171 (delta 65), pack-reused 0 (from 0)
Receiving objects: 100% (205/205), 537.69 KiB | 13.44 MiB/s, done.
Resolving deltas: 100% (99/99), done.
/content/tris
=== Setup GRPO Strict Generation (Colab) ===

📦 Installazione uv...
✅ uv installato

📦 Installazione dipendenze + progetto...
Using Python 3.12.13 environment at: /usr
Resolved 282 packages in 617ms                                       
Prepared 1 package in 331ms                                              
Uninstalled 1 package in 0.40ms
Installed 1 package in 0.96ms==0.1.0 (from file:///content/t
 ~ grpo-strict-generation==0.1.0 (from file:///content/tris)
✅ Dipendenze installate + progetto in editable mode

🔍 Verifica installazione...

AMBIENTE GRPO STRICT GENERATION

🔥 PyTorch:
   Version: 2.10.0+cu128
   CUDA available: True
  

## 2. Imports & GPU Check

In [3]:
from unsloth import FastLanguageModel

print("Unsloth importato correttamente")

import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:             {torch.cuda.get_device_name(0)}")
    print(
        f"VRAM:            {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB"
    )
else:
    print(
        "WARNING: Nessuna GPU trovata! Il training sarà molto lento."
    )

import accelerate
import peft
import transformers
import trl

import datasets as hf_datasets

print(f"transformers: {transformers.__version__}")
print(f"trl:          {trl.__version__}")
print(f"peft:         {peft.__version__}")
print(f"datasets:     {hf_datasets.__version__}")
print(f"accelerate:   {accelerate.__version__}")
print("\nTutti gli import OK")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth importato correttamente
PyTorch version: 2.10.0+cu128
CUDA available:  True
GPU:             Tesla T4
VRAM:            15.6 GB
transformers: 4.57.6
trl:          0.24.0
peft:         0.18.1
datasets:     4.3.0
accelerate:   1.13.0

Tutti gli import OK


## 3. Genera / Carica Dataset Sintetico

In [4]:
import os
from pathlib import Path

from src.datasets.dataloader import load_synthetic_dataset
from src.datasets.synthetic_dataset import generate_dataset

ROOT = Path.cwd()  # la root del progetto è la cartella corrente

os.chdir(ROOT)  # assicurati di essere nella root

DATASET_PATH = ROOT / "data" / "synthetic"

if not DATASET_PATH.exists():
    print("Dataset non trovato, lo genero...")
    ds = generate_dataset(
        num_samples=500, seed=42
    )  # 500 per test veloce
    ds.save_to_disk(str(DATASET_PATH))
    print(f"Dataset salvato in {DATASET_PATH}")
else:
    print(f"Dataset trovato in {DATASET_PATH}")

ds = load_synthetic_dataset(str(DATASET_PATH))
print(f"\nSplits: {list(ds.keys())}")
for split_name, split_ds in ds.items():
    print(f"  {split_name}: {len(split_ds)} samples")
    print(f"    columns: {split_ds.column_names}")

Dataset non trovato, lo genero...


Saving the dataset (0/1 shards):   0%|          | 0/400 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/100 [00:00<?, ? examples/s]

Dataset salvato in /content/tris/data/synthetic

Splits: ['train', 'test']
  train: 400 samples
    columns: ['system_prompt', 'prompt', 'task_type', 'difficulty']
  test: 100 samples
    columns: ['system_prompt', 'prompt', 'task_type', 'difficulty']


In [ ]:
# Guarda qualche sample
train_ds = ds["train"]
for i in range(3):
    sample = train_ds[i]
    print(f"--- Sample {i} ---")
    print(f"  difficulty: {sample['difficulty']}")
    print(f"  prompt:     {sample['prompt'][:120]}...")
    print()

--- Sample 0 ---
  task_type:  json
  difficulty: hard
  prompt:     Generate a JSON object representing a full user management API endpoint specification (OpenAPI-style) including path, me...

--- Sample 1 ---
  task_type:  json
  difficulty: simple
  prompt:     Generate a valid JSON object with the following keys: "category" (string), "count" (integer), "enabled" (boolean)....

--- Sample 2 ---
  task_type:  json
  difficulty: medium
  prompt:     Generate a JSON object representing a specification with the following sections: "metadata", "body", and "status". Each ...



## 4. Carica Modello con Unsloth + LoRA + 4-bit

In [ ]:
from src.utils.config import load_config

config = load_config(
    str(ROOT / "experiments" / "configs" / "grpo_colab.yaml")
)
print("Config caricata:")
for section, values in config.items():
    print(f"  [{section}]")
    if isinstance(values, dict):
        for k, v in values.items():
            print(f"    {k}: {v}")

Config caricata:
  [model]
    name: Qwen/Qwen2.5-Coder-1.5B-Instruct
    quantization: 4bit
    dtype: bfloat16
    use_unsloth: True
  [lora]
    r: 16
    lora_alpha: 32
    lora_dropout: 0.05
    target_modules: ['q_proj', 'k_proj', 'v_proj', 'o_proj']
    task_type: CAUSAL_LM
  [dataset]
    path: data/synthetic
    split: train
    max_samples: None
  [training]
    max_steps: 1000
    per_device_train_batch_size: 1
    gradient_accumulation_steps: 8
    learning_rate: 5e-06
    lr_scheduler_type: cosine
    warmup_ratio: 0.1
    optim: paged_adamw_8bit
    max_grad_norm: 0.1
    bf16: True
    logging_steps: 10
    save_steps: 200
    save_total_limit: 3
    output_dir: experiments/checkpoints/grpo
    log_dir: experiments/logs/grpo
  [grpo]
    num_generations: 8
    max_completion_length: 512
    max_prompt_length: 256
    beta: 0.04
    temperature: 0.7
  [reward]
    partial_credit: True
    reasoning_bonus: 0.0
  [wandb]
    project: grpo-strict-generation
    run_name: Non

In [7]:
from src.models.model_loader import load_model_and_tokenizer

# Forza Unsloth nel config
config["model"]["use_unsloth"] = True

print(f"Caricamento modello con Unsloth: {config['model']['name']}")
print("(questo può richiedere qualche minuto al primo download...)\n")

model, tokenizer = load_model_and_tokenizer(config)

# Info modello
trainable = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)
total = sum(p.numel() for p in model.parameters())
print(f"\n[Unsloth] Parametri totali:    {total:,}")
print(
    f"[Unsloth] Parametri trainable: {trainable:,} ({100 * trainable / total:.2f}%)"
)
print(f"Tokenizer vocab:     {len(tokenizer)}")
print(
    f"Pad token:           {tokenizer.pad_token} (id={tokenizer.pad_token_id})"
)

Caricamento modello con Unsloth: Qwen/Qwen2.5-Coder-1.5B-Instruct
(questo può richiedere qualche minuto al primo download...)

==((====))==  Unsloth 2026.3.15: Fast Qwen2 patching. Transformers: 4.57.6. vLLM: 0.18.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/632 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.3.15 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.



[Unsloth] Parametri totali:    892,974,592
[Unsloth] Parametri trainable: 4,358,144 (0.49%)
Tokenizer vocab:     151666
Pad token:           <|PAD_TOKEN|> (id=151665)


## 5. Test Reward Functions

In [8]:
from src.training.rewards import (
    combined_reward,
    json_reward,
    reasoning_reward,
)

# Test JSON reward
valid_json = (
    '```json\n{"name": "test", "age": 25, "active": true}\n```'
)
invalid_json = '```json\n{"name": "test", age: 25}\n```'
no_json = "Non ho generato nessun JSON."

print("=== JSON Reward ===")
print(f"  Valid JSON:   {json_reward(valid_json)}")  # atteso: 1.0
print(f"  Invalid JSON: {json_reward(invalid_json)}")  # atteso: 0.0
print(f"  No JSON:      {json_reward(no_json)}")  # atteso: 0.0

# Test con partial credit
print("\n=== JSON Reward (partial credit) ===")
print(
    f"  Valid JSON:   {json_reward(valid_json, partial_credit=True)}"
)  # atteso: 1.0
print(
    f"  Invalid JSON: {json_reward(invalid_json, partial_credit=True)}"
)  # atteso: 0.0 o 0.5

# Test reasoning bonus
with_think = (
    "<think>Devo creare un JSON con tre campi come richiesto.</think>\n"
    + valid_json
)
without_think = valid_json

print("\n=== Reasoning Reward ===")
print(
    f"  With <think>:    {reasoning_reward(with_think)}"
)  # atteso: 0.2
print(
    f"  Without <think>: {reasoning_reward(without_think)}"
)  # atteso: 0.0

# Test combined (solo JSON)
print("\n=== Combined Reward (JSON) ===")
print(
    f"  JSON valid:           {combined_reward(valid_json, 'json')}"
)  # 1.0
r = combined_reward(with_think, "json", reasoning_bonus=0.2)
print(f"  JSON valid + reason:  {r}")  # 1.2
print(
    f"  JSON invalid:         {combined_reward(invalid_json, 'json')}"
)  # 0.0

print("\nReward functions OK")

=== JSON Reward ===
  Valid JSON:   1.0
  Invalid JSON: 0.0
  No JSON:      0.0

=== JSON Reward (partial credit) ===
  Valid JSON:   1.0
  Invalid JSON: 0.5

=== Reasoning Reward ===
  With <think>:    0.2
  Without <think>: 0.0

=== Combined Reward (JSON) ===
  JSON valid:           1.0
  JSON valid + reason:  1.2
  JSON invalid:         0.0

Reward functions OK


## 6. Test Inference (pre-training)

In [ ]:
from src.datasets.dataloader import format_prompt_for_model

# Prendi 4 sample (tutti JSON ora)
test_indices = [0, 1, 10, 20]
test_samples = [
    train_ds[i] for i in test_indices if i < len(train_ds)
]

print(f"Genero completions per {len(test_samples)} prompt...\n")

for sample in test_samples:
    prompt = format_prompt_for_model(sample, tokenizer)
    inputs = tokenizer(
        prompt, return_tensors="pt", truncation=True, max_length=512
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.7,
            top_p=0.95,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )

    input_len = inputs["input_ids"].shape[1]
    completion = tokenizer.decode(
        outputs[0][input_len:], skip_special_tokens=True
    )

    # Valuta (tutti JSON)
    difficulty = sample["difficulty"]
    reward = combined_reward(completion)

    print(f"--- [json/{difficulty}] ---")
    print(f"Prompt:     {sample['prompt'][:100]}...")
    print(f"Completion: {completion[:200]}")
    print(f"Reward:     {reward}")
    print()

Genero completions per 4 prompt...

--- [json/hard] ---
Prompt:     Generate a JSON object representing a full user management API endpoint specification (OpenAPI-style...
Completion: ```json
{
  "openapi": "3.0.1",
  "info": {
    "title": "User Management API",
    "version": "1.0"
  },
  "paths": {
    "/users":
      {
        "post":
          {
            "summary": "Create 
Reward:     0.0

--- [json/simple] ---
Prompt:     Generate a valid JSON object with the following keys: "category" (string), "count" (integer), "enabl...
Completion: ```json
{
  "category": "example",
  "count": 5,
  "enabled": true
}
```
Reward:     1.0

--- [json/simple] ---
Prompt:     Generate a valid JSON object with the following keys: "city" (string), "count" (integer), "enabled" ...
Completion: ```json
{
  "city": "Los Angeles",
  "count": 1000,
  "enabled": true
}
```
Reward:     1.0

--- [json/simple] ---
Prompt:     Generate a JSON object with exactly 4 key-value pairs where all values are of typ

## 7. Mini GRPO Training con Unsloth (pochi step di test)

Facciamo un training ridotto per verificare che la pipeline giri senza errori.

- **max_steps=20** — solo per test
- **num_generations=4** — minimo per avere varianza nelle reward
- **learning_rate=5e-6** — come da notebook ufficiali Unsloth
- **beta=0.1** — KL penalty più alto per restare vicini al modello base
- **optim=paged_adamw_8bit** — meno VRAM
- **50 samples** — subset piccolo

### Logging con wandb
Il training logga metriche (reward, KL, loss) su [Weights & Biases](https://wandb.ai).
Su Colab la prima volta chiede l'API key (prendi da https://wandb.ai/authorize).
Per saltare wandb, imposta `report_to="none"` nella cella GRPOConfig.

In [1]:
import wandb

# Login a wandb — necessario su Colab (la prima volta chiede l'API key).
# Prendi la tua key da https://wandb.ai/authorize
# Se vuoi saltare wandb, imposta report_to="none" nella cella GRPOConfig.
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: cosmico445 (cosmico445-cosmic-inc) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
from datasets import Dataset
from src.training.rewards import build_reward_function

# Usa solo 50 samples per il test
MINI_N = 50
mini_ds = train_ds.select(range(min(MINI_N, len(train_ds))))  # type: ignore[arg-type]

# Formatta i prompt
formatted = []
for i in range(len(mini_ds)):
    s = mini_ds[i]
    p = format_prompt_for_model(s, tokenizer)
    formatted.append({"prompt": p})

prompt_dataset = Dataset.from_list(formatted)
print(f"Mini dataset: {len(prompt_dataset)} prompt pronti")

# Reward function
reward_fn = build_reward_function(config["reward"])
print("Reward function creata")

Mini dataset: 50 prompt pronti
Reward function creata


In [11]:
import datetime

from trl import GRPOConfig

# Config leggera per test
mini_output_dir = str(
    ROOT / "experiments" / "checkpoints" / "grpo_test_fast"
)
mini_log_dir = str(ROOT / "experiments" / "logs" / "grpo_test_fast")
Path(mini_output_dir).mkdir(parents=True, exist_ok=True)
Path(mini_log_dir).mkdir(parents=True, exist_ok=True)

_ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
run_name = f"grpo-test-unsloth-{_ts}"

# bf16 richiede GPU Ampere+ (compute capability >= 8.0);
# su GPU più vecchie (es. T4/V100) si ricade su fp16.
_supports_bf16 = (
    torch.cuda.is_available()
    and torch.cuda.get_device_capability()[0] >= 8
)
use_bf16 = _supports_bf16
use_fp16 = not _supports_bf16 and torch.cuda.is_available()
print(
    f"bf16={use_bf16}, fp16={use_fp16}  "
    f"(GPU compute capability: {torch.cuda.get_device_capability() if torch.cuda.is_available() else 'N/A'})"
)

grpo_config = GRPOConfig(  # type: ignore[call-arg]
    output_dir=mini_output_dir,
    run_name=run_name,
    # Training ridotto
    max_steps=20,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,  # Increase to 4 for smoother training
    learning_rate=5e-6,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    optim="paged_adamw_8bit",  # 8-bit optimizer — meno VRAM
    max_grad_norm=0.1,
    bf16=use_bf16,
    fp16=use_fp16,
    # GRPO
    num_generations=4,  # minimo per avere varianza nelle reward
    max_completion_length=256,
    max_prompt_length=256,
    beta=0.1,  # KL penalty più alto per evitare divergenza in pochi step
    temperature=0.7,
    # logging
    logging_dir=mini_log_dir,
    logging_steps=1,
    report_to="wandb",
)

print("GRPOConfig creata")
print(f"  run_name:           {grpo_config.run_name}")
print(f"  max_steps:          {grpo_config.max_steps}")
print(f"  num_generations:    {grpo_config.num_generations}")
print(
    f"  batch_size:         {grpo_config.per_device_train_batch_size}"
)
print(
    f"  grad_accum:         {grpo_config.gradient_accumulation_steps}"
)
print(f"  learning_rate:      {grpo_config.learning_rate}")
print(f"  optim:              {grpo_config.optim}")
print(f"  beta (KL penalty):  {grpo_config.beta}")
print(f"  logging_steps:      {grpo_config.logging_steps}")
print(f"  report_to:          {grpo_config.report_to}")

bf16=False, fp16=True  (GPU compute capability: (7, 5))
Unsloth: We now expect `per_device_train_batch_size` * `gradient_accumulation_steps` * `world_size` to be a multiple of `num_generations`.
We will change the batch size of 1 to the `num_generations` of 4
GRPOConfig creata
  run_name:           grpo-test-unsloth-20260326_224617
  max_steps:          20
  num_generations:    4
  batch_size:         4
  grad_accum:         1
  learning_rate:      5e-06
  optim:              OptimizerNames.PAGED_ADAMW_8BIT
  beta (KL penalty):  0.1
  logging_steps:      1
  report_to:          ['wandb']


In [12]:
from trl.trainer.grpo_trainer import GRPOTrainer

trainer = GRPOTrainer(  # type: ignore[arg-type]
    model=model,
    args=grpo_config,
    train_dataset=prompt_dataset,
    reward_funcs=reward_fn,
    processing_class=tokenizer,
)

print("GRPOTrainer inizializzato")
print(f"  Tipo effettivo: {type(trainer).__name__}")
print("\nAvvio mini-training (20 step)...\n")

trainer.train()
print("\nMini-training completato")

GRPOTrainer inizializzato
  Tipo effettivo: UnslothGRPOTrainer

Avvio mini-training (20 step)...



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 50 | Num Epochs = 1 | Total steps = 20
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 1 x 1) = 4
 "-____-"     Trainable parameters = 4,358,144 of 1,548,072,448 (0.28% trained)


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / reward_fn / mean,rewards / reward_fn / std
1,-0.000000,1.000000,0.000000,33.000000,27.000000,36.000000,0.000000,33.000000,27.000000,36.000000,-0.000000,1.000000,0.000000
2,-0.000000,1.000000,0.000000,50.750000,50.000000,51.000000,0.000000,50.750000,50.000000,51.000000,-0.000000,1.000000,0.000000
3,-0.000000,1.000000,0.000000,188.250000,155.000000,236.000000,0.000000,188.250000,155.000000,236.000000,-0.000000,1.000000,0.000000
4,0.000000,1.000000,0.000000,84.250000,72.000000,95.000000,0.000000,84.250000,72.000000,95.000000,0.000005,1.000000,0.000000
5,0.000000,1.000000,0.000000,28.250000,27.000000,29.000000,0.000000,28.250000,27.000000,29.000000,0.000001,1.000000,0.000000
6,0.000000,0.000000,0.000000,256.000000,256.000000,256.000000,1.000000,0.000000,0.000000,0.000000,0.000006,0.000000,0.000000
7,0.000000,1.000000,0.000000,193.500000,191.000000,200.000000,0.000000,193.500000,191.000000,200.000000,0.000003,1.000000,0.000000
8,0.000000,1.000000,0.000000,39.250000,37.000000,42.000000,0.000000,39.250000,37.000000,42.000000,0.000004,1.000000,0.000000
9,0.000000,1.000000,0.000000,36.500000,34.000000,38.000000,0.000000,36.500000,34.000000,38.000000,0.000008,1.000000,0.000000
10,0.000000,1.000000,0.000000,17.750000,14.000000,19.000000,0.000000,17.750000,14.000000,19.000000,0.000001,1.000000,0.000000


Unsloth: Will smartly offload gradients to save VRAM!

Mini-training completato


## 8. Verifica post-training

In [ ]:
# Rigenera sugli stessi prompt di prima per vedere se qualcosa è cambiato
print("Completions dopo il mini-training:\n")

for sample in test_samples:
    prompt = format_prompt_for_model(sample, tokenizer)
    inputs = tokenizer(
        prompt, return_tensors="pt", truncation=True, max_length=512
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.7,
            top_p=0.95,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )

    input_len = inputs["input_ids"].shape[1]
    completion = tokenizer.decode(
        outputs[0][input_len:], skip_special_tokens=True
    )

    difficulty = sample["difficulty"]
    reward = combined_reward(completion)

    print(f"--- [json/{difficulty}] ---")
    print(f"Prompt:     {sample['prompt'][:100]}...")
    print(f"Completion: {completion[:200]}")
    print(f"Reward:     {reward}")
    print()

Completions dopo il mini-training:

--- [json/hard] ---
Prompt:     Generate a JSON object representing a full user management API endpoint specification (OpenAPI-style...
Completion: ```json
{
  "paths": {
    "/users": {
      "get": {
        "summary": "Get all users",
        "parameters": [
          {
            "name": "page",
            "in": "query",
            "descri
Reward:     0.0

--- [json/simple] ---
Prompt:     Generate a valid JSON object with the following keys: "category" (string), "count" (integer), "enabl...
Completion: ```json
{
  "category": "Electronics",
  "count": 5,
  "enabled": true
}
```
Reward:     1.0

--- [json/simple] ---
Prompt:     Generate a valid JSON object with the following keys: "city" (string), "count" (integer), "enabled" ...
Completion: ```json
{
  "city": "New York",
  "count": 500,
  "enabled": true
}
```
Reward:     1.0

--- [json/simple] ---
Prompt:     Generate a JSON object with exactly 4 key-value pairs where all values are of typ

In [ ]:
# Pulizia VRAM
del trainer
del model
torch.cuda.empty_cache()
print("VRAM liberata")